<a href="https://colab.research.google.com/github/ABugDrone/Machine-Learning-Projects/blob/main/Copy_of_Project_3_Object_Tracking_in_Video_Feeds_for_Traffic_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Mount Google Drive and Unzip the Dataset

In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define path to your dataset
dataset_path = '/content/drive/MyDrive/archive.zip'

# Unzip the dataset
with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall('/content/traffic_data')

# Check the contents
os.listdir('/content/traffic_data')


Mounted at /content/drive


['trafficnet_dataset_v1']

Step 2: Install Dependencies

We'll use YOLOv5 for object detection and Deep SORT for tracking.

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q yolov5
!pip install -q deep-sort-realtime
!pip install opencv-python-headless


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.5/953.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 23.4 MB/s eta 0:00:00


Step 3: Import Libraries and Setup YOLOv5 Model

In [ ]:
import torch
import cv2
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLOv5 model (we'll use a pre-trained version)
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')  # Small version for speed; use 'yolov5m' or 'yolov5l' for better accuracy

# Initialize Deep SORT
deepsort = DeepSort(max_age=30)


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-11-27 Python-3.10.12 torch-2.5.1+cu121 CPU

100%|██████████| 14.1M/14.1M [00:00<00:00, 114MB/s] 

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Step 4: Define Vehicle Detection and Tracking

In [ ]:
def detect_and_track(video_path):
    # Open video file
    cap = cv2.VideoCapture(video_path)

    # Check if video is opened successfully
    if not cap.isOpened():
        print("Error: Unable to open video file.")
        return

    vehicle_count = 0
    frames = []  # List to store processed frames for visualization

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video or error reading frame.")
            break

        # Perform object detection using YOLOv5
        results = model(frame)
        detections = results.pandas().xywh[0]  # Get detection results as pandas dataframe

        # Filter for vehicles (usually with class IDs for car, truck, bus)
        vehicle_detections = detections[detections['class'].isin([2, 3, 5, 7])]  # Class IDs for vehicle types

        # Prepare detections for Deep SORT (x1, y1, x2, y2, confidence)
        bboxes = []
        confidences = []
        for _, row in vehicle_detections.iterrows():
            x1, y1, x2, y2 = row['xmin'], row['ymin'], row['xmax'], row['ymax']
            confidence = row['confidence']
            bboxes.append([x1, y1, x2, y2])
            confidences.append(confidence)

        # Track the detected vehicles using Deep SORT
        tracks = deepsort.update_tracks(bboxes, confidences, frame)

        # Count vehicles passing through a region (lane)
        for track in tracks:
            if track.is_confirmed() and track.time_since_update <= 1:
                vehicle_count += 1  # Count vehicles

        # Display the results on the frame (for debugging)
        for track in tracks:
            if track.is_confirmed():
                bbox = track.to_tlbr()  # Get bounding box coordinates
                cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
                cv2.putText(frame, f'ID: {track.track_id}', (int(bbox[0]), int(bbox[1]-10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Convert BGR to RGB for displaying in Colab
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)  # Save the frame for later visualization

    cap.release()
    print(f"Total vehicle count: {vehicle_count}")

    if frames:  # Check if there are frames to display
        # Show the first frame (for visualization)
        plt.imshow(frames[0])
        plt.axis('off')
        plt.show()

        # Optionally, you can show the video as a sequence of frames
        for frame in frames[:10]:  # Show the first 10 frames (adjust as needed)
            plt.imshow(frame)
            plt.axis('off')
            plt.pause(0.1)  # Pause to simulate frame rate
    else:
        print("No frames were processed.")

# Define the path to your video file
video_path = '/content/traffic_data/video.mp4'  # Adjust based on your dataset
detect_and_track(video_path)


Error: Unable to open video file.


In [ ]:
import os
os.listdir('/content/traffic_data')


['trafficnet_dataset_v1']

In [ ]:
import os
# List files in the folder to ensure the video file exists
video_folder = '/content/traffic_data'
os.listdir(video_folder)


['trafficnet_dataset_v1']

In [ ]:
video_path = '/content/traffic_data/traffic_video.mp4'


In [ ]:
!ffmpeg -i /content/traffic_data/your_video_file.mov /content/traffic_data/traffic_video.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
!ffmpeg -i /content/traffic_data/your_video_file.mov /content/traffic_data/traffic_video.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
!apt-get install -y ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


In [ ]:
import cv2

video_path = '/content/traffic_data/traffic_video.mp4'  # Update the path as needed

cap = cv2.VideoCapture(video_path)

# Check if video is opened successfully
if not cap.isOpened():
    print("Error: Unable to open video file.")
else:
    print("Video file opened successfully.")

cap.release()


Error: Unable to open video file.


A camera source needed to fully test code on the street. Capstone 3 successfully attempted.